In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [16]:
url = 'https://quotes.toscrape.com/'
response = requests.get(url)

# parse the html content
soup = BeautifulSoup(response.content,'html.parser')

In [37]:
quotes_elements = soup.find_all("span", class_="text")
author_elements = soup.find_all('small',class_='author')
tag_elements = soup.find_all('div',class_='tags')

# creating lists to store quotes and authors
all_quotes = []
all_authors = []
all_tags = []


for quote, auth, tag_div_element in zip(quotes_elements, author_elements, tag_elements):
    all_quotes.append(quote.text)
    all_authors.append(auth.text)

    # Extract individual tags
    individual_tags = tag_div_element.find_all('a', class_='tag')
    cleaned_tags = [tag.text for tag in individual_tags]
    all_tags.append(cleaned_tags)

# creating the dataframe of quote and author
df = pd.DataFrame({
    'quote': all_quotes,
    'author': all_authors,
    'tags': all_tags
})

print('dataframe is created')
df.head()

dataframe is created


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"


In [43]:
# Initialize master lists to collect data from all pages
master_all_quotes = []
master_all_authors = []
master_all_tags = []

for i in range(2,7):
    url = f'https://quotes.toscrape.com/page/{i}/'
    response = requests.get(url)
    soup = BeautifulSoup(response.content,'html.parser')

    quotes_elements = soup.find_all("span", class_="text")
    author_elements = soup.find_all('small',class_='author')
    tag_elements = soup.find_all('div',class_='tags')

    # Temporary lists for current page's data
    page_quotes = []
    page_authors = []
    page_tags = []

    for quote, auth, tag_div_element in zip(quotes_elements, author_elements, tag_elements):
        page_quotes.append(quote.text)
        page_authors.append(auth.text)

        individual_tags = tag_div_element.find_all('a', class_='tag')
        cleaned_tags = [tag.text for tag in individual_tags]
        page_tags.append(cleaned_tags)

    # Extend master lists with data from the current page
    master_all_quotes.extend(page_quotes)
    master_all_authors.extend(page_authors)
    master_all_tags.extend(page_tags)

# Create the DataFrame after collecting data from all pages
df = pd.DataFrame({
    'quote': master_all_quotes,
    'author': master_all_authors,
    'tags': master_all_tags
})


print('dataset with more data....')
df.shape

dataset with more data....


(50, 3)

In [45]:
# craeting the csv file
df.to_csv('qouites.csv',index=False)
print('csv file is created')

csv file is created


In [46]:
html_layout_monday = """
<div class="product-card">
    <h1 class="item-title">Vintage Leather Jacket</h1>
    <span class="product-price">$89.99</span>
</div>
"""

html_layout_tuesday = """
<section class="item-container">
    <h2 class="main-heading">Vintage Leather Jacket</h2>
    <div class="discounted-cost-wrapper">89.99 USD</div>
</section>
"""

In [47]:
# this function scans the page looking for text patterns that *look like* a price.
import re

def get_price(html_code):
  soup = BeautifulSoup(html_code, 'html.parser')

  page_text = soup.get_text(separator=' ',strip=True)
  print(f"--- Raw Text Extracted --- \n{page_text}")

  price_pattern = r"\$?\d+\.\d{2}"
  # Search the text for our pattern
  match = re.search(price_pattern , page_text)

  if match:
        return match.group()  # Return the price we found
  else:
        return "Price not found"

In [49]:
print("Testing Monday's Layout...")
price_monday = get_price(html_layout_monday)
print(f"Smart Finder Found: {price_monday}\n")

print("---------------------------------")

print("Testing Tuesday's Layout...")
price_tuesday = get_price(html_layout_tuesday)
print(f"Smart Finder Found: {price_tuesday}\n")

Testing Monday's Layout...
--- Raw Text Extracted --- 
Vintage Leather Jacket $89.99
Smart Finder Found: $89.99

---------------------------------
Testing Tuesday's Layout...
--- Raw Text Extracted --- 
Vintage Leather Jacket 89.99 USD
Smart Finder Found: 89.99



In [50]:
# # The Problem with Simple Scrapers
# # If you try to use requests.get() on websites like Twitter,
# infinite-scroll clothing stores, or Javascript-heavy pages,
#  you usually get back an almost empty page. Why? Because requests only downloads the initial blank HTML file.
#   It doesn't run the JavaScript engine required to actually fetch and show the content.

! pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 18.7 MB/s eta 0:00:00


In [58]:
import time
import nest_asyncio
import asyncio # Import asyncio
from playwright.async_api import async_playwright # Use async_playwright
from bs4 import BeautifulSoup

# This patch prevents Playwright from crashing inside Google Colab's notebook engine
nest_asyncio.apply()

async def scrape_dynamic_page_on_colab(): # Define as async function
    async with async_playwright() as p: # Use async with
        print("Launching hidden background browser...")
        # CRITICAL FOR COLAB: headless MUST be True because Colab has no monitor screen attached
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        print("Navigating to the dynamic JavaScript page...")
        await page.goto("https://quotes.toscrape.com/js/")

        # Wait for JavaScript execution
        await asyncio.sleep(2) # Use asyncio.sleep for async

        print("Simulating scroll to the bottom of the page...")
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(2) # Use asyncio.sleep for async

        # Pull the fully rendered HTML out of the browser memory
        html_content = await page.content()
        await browser.close()
        print("Browser closed successfully.")

    # Parse the text just like standard BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")
    quotes = soup.find_all("span", class_="text")

    print("\n--- Extracted Quotes on Colab ---")
    for idx, quote in enumerate(quotes, 1):
        print(f"{idx}. {quote.text}")

# Execute the function using asyncio.run()
asyncio.run(scrape_dynamic_page_on_colab())

Launching hidden background browser...
Navigating to the dynamic JavaScript page...
Simulating scroll to the bottom of the page...
Browser closed successfully.

--- Extracted Quotes on Colab ---
1. “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
2. “It is our choices, Harry, that show what we truly are, far more than our abilities.”
3. “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
4. “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
5. “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
6. “Try not to become a man of success. Rather become a man of value.”
7. “It is better to be hated for what you are than to be loved for what you are not.”
8. “I have not failed. I've just found 10,000 ways that won't work.”
9. “A woman i